# Étude de cas : Hubble en 1929, une loi dans des mesures bruitées

En 1929, Edwin Hubble a publié les distances et les vitesses d'éloignement de 24
nébuleuses, et la proportion entre les deux. Les mesures sont très dispersées :
une droite explique 62 % de leur variance. Ce notebook donne ces 24 lignes à
GP_ELITE. Avec les unités, il rend v = H·d avec H en km/(Mpc·s), la droite par
l'origine que donnent les moindres carrés ; sans les unités, la droite avec une
ordonnée à l'origine. Il ne trouve rien au-delà d'une droite, et c'est ce que
les données permettent.

Chaque chiffre du texte a été mesuré avec `benchmarks/case_studies.py hubble`,
pour les graines 0 à 4 (sortie dans
`benchmarks/results_0.9/case_studies/hubble.txt`).

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Chaque ajustement
prend de dix à vingt secondes.

*[🇬🇧 English version](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/hubble_1929.ipynb)*

## 1. Installation

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'prêt.')

## 2. Les données et leur source

Le tableau vient du paquet R Sleuth2 (jeu `case0701` ; Ramsey et Schafer,
*The Statistical Sleuth*, 2e édition, étude de cas 7.1), qui transcrit celui de
Hubble (*Proceedings of the National Academy of Sciences* 15, 168, 1929) :
distance en mégaparsecs, vitesse en km/s. Une seconde transcription, dans le
paquet R lmreg, donne −200 km/s au lieu de +200 pour la nébuleuse à 0,63 Mpc ;
nous n'avons pas pu consulter le tableau d'origine pour trancher, et la section
6 montre ce que cela change.

In [ ]:
import os
import numpy as np
import pandas as pd

NAME = 'hubble1929.csv'
URL = ('https://raw.githubusercontent.com/ariel95500-create/gp-elite/main/'
       'examples/case_studies/data/' + NAME)
path = os.path.join('data', NAME)
df = pd.read_csv(path if os.path.exists(path) else URL, comment='#')

d = df['distance_mpc'].to_numpy(float)
v = df['velocity_kms'].to_numpy(float)
print(df.to_string(index=False))

## 3. Les droites à battre

Avant toute recherche, les deux droites des moindres carrés : par l'origine,
comme le veut la loi de Hubble, et avec une ordonnée à l'origine.

In [ ]:
def r2(p, t):
    return 1 - np.mean((p - t) ** 2) / np.var(t)

def rmse(p, t):
    return np.sqrt(np.mean((p - t) ** 2))

k0 = d @ v / (d @ d)              # par l'origine
b1, b0 = np.polyfit(d, v, 1)      # avec une ordonnée à l'origine
print(f"par l'origine             : v = {k0:.2f} d             "
      f'R2 {r2(k0 * d, v):.4f}   erreur {rmse(k0 * d, v):.1f} km/s')
print(f"avec ordonnée à l'origine : v = {b0:.2f} + {b1:.2f} d   "
      f'R2 {r2(b0 + b1 * d, v):.4f}   erreur {rmse(b0 + b1 * d, v):.1f} km/s')

Par l'origine, v = 423,94·d, R² 0,6194, erreur quadratique moyenne 224 km/s.
Avec une ordonnée à l'origine, v = −40,78 + 454,16·d, R² 0,6235, 223 km/s.

## 4. GP_ELITE, avec et sans les unités

In [ ]:
from gp_elite import GPEliteRegressor

X = d.reshape(-1, 1)

with_units = GPEliteRegressor(units=['Mpc'], target_units='km/s',
                              unknown_constant=True, generations=30,
                              speed='fast', random_state=0, parallel=False)
with_units.fit(X, v)
p = with_units.predict(X)
print('avec unités :', with_units.equation_)
print('constante   :', with_units.constant_value_, with_units.constant_units_string())
print(f'              R2 {r2(p, v):.4f}   erreur {rmse(p, v):.1f} km/s')

no_units = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                            parallel=False)
no_units.fit(X, v)
p = no_units.predict(X)
print('sans unités :', no_units.equation_)
print(f'              R2 {r2(p, v):.4f}   erreur {rmse(p, v):.1f} km/s')

Avec les unités, le moteur rend `423.937 * X0` : la droite par l'origine des
moindres carrés, et sa constante en km/(Mpc·s), les unités de la constante de
Hubble. Une vitesse ne peut pas être un simple nombre : avec les unités, la
droite ne peut pas prendre d'ordonnée à l'origine, et passe par l'origine comme
la loi de Hubble. Sans les unités, il rend `-40.7836 + 454.158 * X0`, la droite
avec ordonnée à l'origine, qui baisse un peu l'erreur (223,0 km/s contre 224,2).
Les graines 0 à 4 rendent toutes ces deux formules.

La constante, 424 km/(Mpc·s), est la constante de Hubble telle que ces données
la donnent. On a découvert plus tard que les distances de Hubble étaient
beaucoup trop courtes, et la valeur mesurée aujourd'hui est plusieurs fois plus
petite ; aucun ajustement de ces 24 lignes ne peut le voir.

**Ce que font ici les graines tolérantes au bruit.** Cette version du moteur
place, quand elle ne trouve aucune forme exacte, une forme qui explique les
données à 1,5 % de leur variance près. Ici la meilleure droite en laisse 38 % :
ces graines ne placent rien, et les graines exactes non plus (sans les unités ;
avec les unités, aucune d'elles n'agit). Ce que vous voyez est la recherche
seule, qui trouve la droite par elle-même.

## 5. Une vérification tenue à l'écart : les six nébuleuses les plus lointaines

Les deux ajustements de nouveau, sur les 18 nébuleuses jusqu'à 1,1 Mpc, pour
prédire les 6 au-delà (de 1,4 à 2,0 Mpc).

In [ ]:
near, far = d <= 1.1, d > 1.1
k_near = d[near] @ v[near] / (d[near] @ d[near])
print(f'{near.sum()} nébuleuses pour ajuster, {far.sum()} à prédire')
print(f"moindres carrés par l'origine : v = {k_near:.2f} d   "
      f'erreur sur les 6 : {rmse(k_near * d[far], v[far]):.1f} km/s')

for label, kw in (('avec unités', dict(units=['Mpc'], target_units='km/s',
                                       unknown_constant=True)),
                  ('sans unités', {})):
    m = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False, **kw)
    m.fit(X[near], v[near])
    print(f'{label:11s} : {m.equation_}   '
          f'erreur sur les 6 : {rmse(m.predict(X[far]), v[far]):.1f} km/s')

Avec les unités, `427.063 * X0`, et une erreur de 203 km/s sur les six
nébuleuses lointaines, la même que les moindres carrés par l'origine. Sans les
unités, `-74.9921 + 520.269 * X0`, et 231 km/s. Ici la loi par l'origine porte
plus loin ; sur six lignes aussi dispersées, on n'en tirerait pas davantage.

## 6. L'autre transcription

Les deux mêmes ajustements, avec −200 km/s pour la nébuleuse à 0,63 Mpc.

In [ ]:
v_lmreg = v.copy()
v_lmreg[9] = -200.0     # la dixième ligne, 0,63 Mpc, comme dans le paquet R lmreg

for label, kw in (('avec unités', dict(units=['Mpc'], target_units='km/s',
                                       unknown_constant=True)),
                  ('sans unités', {})):
    m = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False, **kw)
    m.fit(X, v_lmreg)
    print(f'{label:11s} : {m.equation_}   R2 {r2(m.predict(X), v_lmreg):.4f}')

Avec les unités, `415.4 * X0` : une ligne sur 24 déplace la constante de 2 %.
Sans les unités, `-68.1539 + 465.903 * X0`.

## 7. Ce que le moteur ne trouve pas

- **Quoi que ce soit au-delà d'une droite.** Sur ces 24 lignes, le moteur rend
  la droite, avec ou sans ordonnée à l'origine, et rien de plus.
- **Pourquoi les points se dispersent.** Les erreurs sur les distances et les
  mouvements propres des nébuleuses ne sont pas dans les données.
- **La vraie valeur de H.** Le moteur ajuste les données qu'on lui donne ; savoir
  si les distances sont justes est une question d'astronomes.

## 8. Les cinq graines

Les graines 0 à 4 rendent les mêmes formules, au dernier chiffre affiché près,
dans chaque ajustement de ce notebook. Les chiffres ont été mesurés sur la
branche de développement qui suit la version 0.9.0, puis de nouveau sur le
paquet des versions 1.0.0 et 1.1.0, avec les mêmes résultats au dernier chiffre
affiché près (`benchmarks/results_0.9/release_1.0/m3/`,
`benchmarks/results_1.1/release/m3/`).

La sortie complète : `benchmarks/results_0.9/case_studies/hubble.txt`. Le
protocole, écrit avant les mesures : `benchmarks/results_0.9/PLAN_PHASE6.md`.